In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [2]:
spark = SparkSession.builder.appName("SparkOptimization").getOrCreate() 

In [5]:
data = [(1, "2025-01-01", 100),
        (1, "2025-01-02", 200),
        (1, "2025-01-03", 150), 
        (2, "2025-01-01", 300),
        (2, "2025-01-02", 250)
        ]  
order_df= spark.createDataFrame(data, 
                                [ "CustomerID", "OrderDate", "SalesAmount"])

In [4]:
from pyspark.sql.window import Window

In [7]:
order_df

DataFrame[CustomerID: bigint, OrderDate: string, SalesAmount: bigint]

In [8]:
order_df = order_df.withColumn("OrderDate",to_date("OrderDate"))

In [10]:
order_df

DataFrame[CustomerID: bigint, OrderDate: date, SalesAmount: bigint]

In [16]:
window_spec = Window.partitionBy("CustomerID").orderBy(("OrderDate"))

In [17]:
running_sum_df = order_df.withColumn("running_sum", sum("SalesAmount").over(window_spec) )

In [18]:
spark.conf.set("spark.sql.shuffle.partitions",8)

In [19]:
running_sum_df.show()

+----------+----------+-----------+-----------+
|CustomerID| OrderDate|SalesAmount|running_sum|
+----------+----------+-----------+-----------+
|         1|2025-01-01|        100|        100|
|         1|2025-01-02|        200|        300|
|         1|2025-01-03|        150|        450|
|         2|2025-01-01|        300|        300|
|         2|2025-01-02|        250|        550|
+----------+----------+-----------+-----------+



In [20]:
data = [(1, "2025-01-01", 100),
        (1, "2025-01-02", 200),
        (1, "2025-01-02", 100),
        (1, "2025-01-03", 150), 
        (2, "2025-01-01", 300),
        (2, "2025-01-02", 250)
        ]  
order_df= spark.createDataFrame(data, 
                                [ "CustomerID", "OrderDate", "SalesAmount"])

In [21]:
window_spec = Window.partitionBy("CustomerID").orderBy(desc("OrderDate"))

In [22]:
row_num  = order_df.withColumn("row_num", row_number().over(window_spec))

In [23]:
row_num.show()

+----------+----------+-----------+-------+
|CustomerID| OrderDate|SalesAmount|row_num|
+----------+----------+-----------+-------+
|         1|2025-01-03|        150|      1|
|         1|2025-01-02|        200|      2|
|         1|2025-01-02|        100|      3|
|         1|2025-01-01|        100|      4|
|         2|2025-01-02|        250|      1|
|         2|2025-01-01|        300|      2|
+----------+----------+-----------+-------+



In [25]:
row_num.filter("row_num = 1").show()

+----------+----------+-----------+-------+
|CustomerID| OrderDate|SalesAmount|row_num|
+----------+----------+-----------+-------+
|         1|2025-01-03|        150|      1|
|         2|2025-01-02|        250|      1|
+----------+----------+-----------+-------+



In [26]:
rank_n_drnk  = order_df.withColumn("rank_clm", rank().over(window_spec))\
                .withColumn("drank_clm", dense_rank().over(window_spec))

In [27]:
rank_n_drnk.show()

+----------+----------+-----------+--------+---------+
|CustomerID| OrderDate|SalesAmount|rank_clm|drank_clm|
+----------+----------+-----------+--------+---------+
|         1|2025-01-03|        150|       1|        1|
|         1|2025-01-02|        200|       2|        2|
|         1|2025-01-02|        100|       2|        2|
|         1|2025-01-01|        100|       4|        3|
|         2|2025-01-02|        250|       1|        1|
|         2|2025-01-01|        300|       2|        2|
+----------+----------+-----------+--------+---------+



In [28]:
data = [(1, "2025-01-01", 100),
        (1, "2025-01-02", 200),
        (1, "2025-01-03", 150), 
        (2, "2025-01-01", 300),
        (2, "2025-01-02", 250)
        ]  
order_df= spark.createDataFrame(data, 
                                [ "CustomerID", "OrderDate", "SalesAmount"])

In [30]:
window_spec = Window.partitionBy("CustomerID").orderBy(("OrderDate"))

In [32]:
lead_n_lag = order_df.withColumn("prev_amount", lag("SalesAmount").over(window_spec))\
                    .withColumn("next_amount", lead("SalesAmount").over(window_spec))

In [33]:
lead_n_lag.show()

+----------+----------+-----------+-----------+-----------+
|CustomerID| OrderDate|SalesAmount|prev_amount|next_amount|
+----------+----------+-----------+-----------+-----------+
|         1|2025-01-01|        100|       NULL|        200|
|         1|2025-01-02|        200|        100|        150|
|         1|2025-01-03|        150|        200|       NULL|
|         2|2025-01-01|        300|       NULL|        250|
|         2|2025-01-02|        250|        300|       NULL|
+----------+----------+-----------+-----------+-----------+



In [34]:
# DataFrame Specification    # UNBOUNDED PRECEEDING TO CURRENT ROW 

In [35]:
window_spec = Window.partitionBy("CustomerID").orderBy(("OrderDate")).rowsBetween(-1,1)

In [37]:
order_df.withColumn("moving_sum", sum("SalesAmount").over(window_spec) ).show()

+----------+----------+-----------+----------+
|CustomerID| OrderDate|SalesAmount|moving_sum|
+----------+----------+-----------+----------+
|         1|2025-01-01|        100|       300|
|         1|2025-01-02|        200|       450|
|         1|2025-01-03|        150|       350|
|         2|2025-01-01|        300|       550|
|         2|2025-01-02|        250|       550|
+----------+----------+-----------+----------+



In [40]:
window_spec = Window.partitionBy("CustomerID").orderBy(desc("SalesAmount"))

In [41]:
order_df.withColumn("top_tx", row_number().over(window_spec)).filter("top_tx = 1").show()

+----------+----------+-----------+------+
|CustomerID| OrderDate|SalesAmount|top_tx|
+----------+----------+-----------+------+
|         1|2025-01-02|        200|     1|
|         2|2025-01-01|        300|     1|
+----------+----------+-----------+------+



In [42]:
#  Group By and Window 
#  Shuffling ; Aggregarion ; reduce num of rec    |     Shuffle ; inside the partiton ; Keep the rows

In [44]:
window_spec = Window.partitionBy("CustomerID")
order_df.withColumn("sum_sales",sum("SalesAmount").over(window_spec))\
        .withColumn("avg_sales",avg("SalesAmount").over(window_spec)).show()

+----------+----------+-----------+---------+---------+
|CustomerID| OrderDate|SalesAmount|sum_sales|avg_sales|
+----------+----------+-----------+---------+---------+
|         1|2025-01-01|        100|      450|    150.0|
|         1|2025-01-02|        200|      450|    150.0|
|         1|2025-01-03|        150|      450|    150.0|
|         2|2025-01-01|        300|      550|    275.0|
|         2|2025-01-02|        250|      550|    275.0|
+----------+----------+-----------+---------+---------+



In [45]:
spark.conf.set("spark.sql.adaptive.enabled","true")

In [47]:
df = spark.range(1, 5000000)
orders_df = df.withColumn("CustomerID", floor(rand()*100000)) \
    .withColumn("RegionId",  floor(rand()*10)) \
    .withColumn("Amount", floor(rand()*1000)) \
    .withColumn("OrderDate",current_date())

In [48]:
orders_df

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint, OrderDate: date]

In [49]:
orders_df.show(5)

+---+----------+--------+------+----------+
| id|CustomerID|RegionId|Amount| OrderDate|
+---+----------+--------+------+----------+
|  1|     95718|       0|    14|2026-02-19|
|  2|     29990|       5|   233|2026-02-19|
|  3|       925|       0|   439|2026-02-19|
|  4|     30148|       3|   363|2026-02-19|
|  5|      9371|       8|   741|2026-02-19|
+---+----------+--------+------+----------+
only showing top 5 rows



In [50]:
# customer_df = order_df.withColumn("CustomerID").distinct()

In [53]:
customer_df = spark.range(0,100000) \
                .withColumnRenamed("id","CustomerId") \
                .withColumn("CustomerName",concat(lit("Cust_"),col("CustomerId"))) 

In [54]:
customer_df.show(5)

+----------+------------+
|CustomerId|CustomerName|
+----------+------------+
|         0|      Cust_0|
|         1|      Cust_1|
|         2|      Cust_2|
|         3|      Cust_3|
|         4|      Cust_4|
+----------+------------+
only showing top 5 rows



In [57]:
from pyspark.sql.functions import broadcast

In [58]:
joined_df = order_df.join(broadcast(customer_df), "CustomerId")

In [59]:
joined_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [CustomerId])
:- LogicalRDD [CustomerID#188L, OrderDate#189, SalesAmount#190L], false
+- ResolvedHint (strategy=broadcast)
   +- Project [CustomerId#436L, concat(Cust_, cast(CustomerId#436L as string)) AS CustomerName#438]
      +- Project [id#434L AS CustomerId#436L]
         +- Range (0, 100000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
CustomerID: bigint, OrderDate: string, SalesAmount: bigint, CustomerName: string
Project [CustomerID#188L, OrderDate#189, SalesAmount#190L, CustomerName#438]
+- Join Inner, (CustomerID#188L = CustomerId#436L)
   :- LogicalRDD [CustomerID#188L, OrderDate#189, SalesAmount#190L], false
   +- ResolvedHint (strategy=broadcast)
      +- Project [CustomerId#436L, concat(Cust_, cast(CustomerId#436L as string)) AS CustomerName#438]
         +- Project [id#434L AS CustomerId#436L]
            +- Range (0, 100000, step=1, splits=Some(8))

== Optimized Logical Plan ==
Project [CustomerID#188L, OrderDate#

In [60]:
from pyspark import StorageLevel
#joined_df.persist()

In [61]:
joined_df.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[CustomerID: bigint, OrderDate: string, SalesAmount: bigint, CustomerName: string]

In [62]:
joined_df.count()

5

In [64]:
orders_df

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint, OrderDate: date]

In [65]:
region_revenue_df = orders_df.groupBy("RegionId").sum("Amount")

In [66]:
region_revenue_df.show()

+--------+-----------+
|RegionId|sum(Amount)|
+--------+-----------+
|       2|  248828520|
|       9|  249909933|
|       3|  249900831|
|       7|  249298817|
|       5|  250287019|
|       8|  249348784|
|       4|  250241806|
|       0|  249858050|
|       6|  249579165|
|       1|  249793026|
+--------+-----------+



In [67]:
region_revenue_df = orders_df.groupBy("RegionId").sum("Amount").alias("Total_Revenue")

In [69]:
region_revenue_df.show()

+--------+-----------+
|RegionId|sum(Amount)|
+--------+-----------+
|       2|  248828520|
|       9|  249909933|
|       3|  249900831|
|       7|  249298817|
|       5|  250287019|
|       8|  249348784|
|       4|  250241806|
|       0|  249858050|
|       6|  249579165|
|       1|  249793026|
+--------+-----------+



In [70]:
region_revenue_df = orders_df.groupBy("RegionId").agg(sum("Amount").alias("Total_Revenue"))

In [71]:
region_revenue_df

DataFrame[RegionId: bigint, Total_Revenue: bigint]

In [72]:
region_revenue_df.show()

+--------+-------------+
|RegionId|Total_Revenue|
+--------+-------------+
|       2|    248828520|
|       9|    249909933|
|       3|    249900831|
|       7|    249298817|
|       5|    250287019|
|       8|    249348784|
|       4|    250241806|
|       0|    249858050|
|       6|    249579165|
|       1|    249793026|
+--------+-------------+



In [75]:
window_spec = Window.orderBy(desc("Total_Revenue"))
region_revenue_df.withColumn("row_num", row_number().over(window_spec)).filter("row_num <= 3").show()

+--------+-------------+-------+
|RegionId|Total_Revenue|row_num|
+--------+-------------+-------+
|       5|    250287019|      1|
|       4|    250241806|      2|
|       9|    249909933|      3|
+--------+-------------+-------+



In [76]:
orders_df

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint, OrderDate: date]

In [77]:
window_spec = Window.partitionBy("CustomerID").orderBy(desc("Amount"))
orders_df.withColumn("row_num", row_number().over(window_spec)).filter("row_num <= 3").show()

+-------+----------+--------+------+----------+-------+
|     id|CustomerID|RegionId|Amount| OrderDate|row_num|
+-------+----------+--------+------+----------+-------+
|2324866|         2|       1|   963|2026-02-19|      1|
|3132368|         2|       8|   936|2026-02-19|      2|
|3155281|         2|       7|   922|2026-02-19|      3|
|1234075|        12|       3|   985|2026-02-19|      1|
|2598949|        12|       4|   959|2026-02-19|      2|
|1312555|        12|       7|   954|2026-02-19|      3|
|1186231|        26|       0|   968|2026-02-19|      1|
|3417352|        26|       7|   964|2026-02-19|      2|
|3613009|        26|       2|   964|2026-02-19|      3|
|2165987|        28|       5|   946|2026-02-19|      1|
| 504745|        28|       2|   907|2026-02-19|      2|
|1206784|        28|       2|   855|2026-02-19|      3|
|4411325|        29|       8|   998|2026-02-19|      1|
|3184992|        29|       0|   959|2026-02-19|      2|
|2470873|        29|       7|   952|2026-02-19| 

In [80]:
# Top 3 customer 
window_spec = Window.orderBy(desc("Amount"))
orders_df.withColumn("row_num", row_number().over(window_spec)).filter("row_num <= 3").show()

+----+----------+--------+------+----------+-------+
|  id|CustomerID|RegionId|Amount| OrderDate|row_num|
+----+----------+--------+------+----------+-------+
| 873|     31407|       7|   999|2026-02-19|      1|
|4896|     78852|       9|   999|2026-02-19|      2|
|2725|     46261|       2|   999|2026-02-19|      3|
+----+----------+--------+------+----------+-------+



In [82]:
# Top 3 customers in each region 
window_spec = Window.partitionBy("RegionId").orderBy(desc("Amount"))

In [86]:
#agg_df = orders_df
orders_df.withColumn("row_number", row_number().over(window_spec)).filter("row_number <= 3").show()

+-----+----------+--------+------+----------+----------+
|   id|CustomerID|RegionId|Amount| OrderDate|row_number|
+-----+----------+--------+------+----------+----------+
|18466|     20399|       0|   999|2026-02-19|         1|
|27531|     94037|       0|   999|2026-02-19|         2|
|27792|     12433|       0|   999|2026-02-19|         3|
|13415|     96618|       1|   999|2026-02-19|         1|
|15368|     43160|       1|   999|2026-02-19|         2|
|16935|     27792|       1|   999|2026-02-19|         3|
| 2725|     46261|       2|   999|2026-02-19|         1|
|21248|     76490|       2|   999|2026-02-19|         2|
|41168|     43179|       2|   999|2026-02-19|         3|
|30712|     49316|       3|   999|2026-02-19|         1|
|43705|     29141|       3|   999|2026-02-19|         2|
|47577|     69029|       3|   999|2026-02-19|         3|
| 9344|     50376|       4|   999|2026-02-19|         1|
|21181|     10853|       4|   999|2026-02-19|         2|
|29106|     48064|       4|   9

In [87]:
agg_df = orders_df.groupBy("CustomerID","RegionId").agg(sum("Amount").alias("Total_Amount"))

In [89]:
# Top3 customer in  each region 
window_spec = Window.partitionBy("RegionId").orderBy(desc("Total_Amount"))
agg_df.withColumn("row_number", row_number().over(window_spec)).filter("row_number <= 3").show()

+----------+--------+------------+----------+
|CustomerID|RegionId|Total_Amount|row_number|
+----------+--------+------------+----------+
|     67729|       0|        9488|         1|
|     94306|       0|        9374|         2|
|     97469|       0|        9217|         3|
|      4572|       1|       10927|         1|
|     46896|       1|       10553|         2|
|     58895|       1|       10024|         3|
|     50825|       2|        9949|         1|
|     87375|       2|        9944|         2|
|     46919|       2|        9675|         3|
|     74555|       3|       10123|         1|
|     82100|       3|       10119|         2|
|     55869|       3|        9522|         3|
|     68973|       4|       10442|         1|
|      2792|       4|        9762|         2|
|     12500|       4|        9672|         3|
|     18404|       5|       11950|         1|
|     95250|       5|       11613|         2|
|     87721|       5|        9820|         3|
|     79457|       6|        9839|

In [90]:
agg_df.withColumn("row_number", row_number().over(window_spec)).filter("row_number <= 3").explain(True)

== Parsed Logical Plan ==
'Filter ('row_number <= 3)
+- Project [CustomerID#379L, RegionId#382L, Total_Amount#929L, row_number#964]
   +- Project [CustomerID#379L, RegionId#382L, Total_Amount#929L, row_number#964, row_number#964]
      +- Window [row_number() windowspecdefinition(RegionId#382L, Total_Amount#929L DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS row_number#964], [RegionId#382L], [Total_Amount#929L DESC NULLS LAST]
         +- Project [CustomerID#379L, RegionId#382L, Total_Amount#929L]
            +- Aggregate [CustomerID#379L, RegionId#382L], [CustomerID#379L, RegionId#382L, sum(Amount#386L) AS Total_Amount#929L]
               +- Project [id#377L, CustomerID#379L, RegionId#382L, Amount#386L, current_date(Some(Asia/Calcutta)) AS OrderDate#391]
                  +- Project [id#377L, CustomerID#379L, RegionId#382L, FLOOR((rand(-6228603909137513099) * cast(1000 as double))) AS Amount#386L]
                     +- Project [id#377L, Cu